# ChakraModel - Final Kaggle Proof Evaluation

This notebook provides the definitive, undeniable proof of ChakraModel's performance on full-cohort OOD datasets and real-world video sequences from HyperKvasir. No truncation. No synthetic data.

## 1. Environment Setup

In [1]:
import os
import stat
import errno

# ==========================================================
# KAGGLE DATASET PATH VALIDATOR
# Paste any list of dataset/folder paths into CANDIDATE_PATHS
# below (or load them from a config/list variable) and run.
# Reports OK / MISSING / error-type for every path so you can
# fix broken attachments before your training script runs.
# ==========================================================

# --- 1. PASTE YOUR PATHS HERE -------------------------------------------
# You can freely add/remove/edit entries. Duplicate entries are
# automatically de-duplicated further down.
CANDIDATE_PATHS = [
    "/kaggle/input/datasets/gokulrocky/chakramodel-evaluation-datasets/kvasir-seg/images",
    "/kaggle/input/datasets/gokulrocky/chakramodel-kaggle-code",
    "/kaggle/input/datasets/gokulrocky/chakratransformer-weights",
    "/kaggle/input/datasets/gokulrocky/cvc-sample-video",
    "/kaggle/input/datasets/gokulrocky/endoscene-cvc300-polyp-raw-dataset",
    "/kaggle/input/datasets/gokulrocky/final-om-evlautation-upload",
    "/kaggle/input/datasets/gokulrocky/finalmuruga-harae",
    "/kaggle/input/datasets/gokulrocky/hperkvasir-labeled-videos-part2-002",
    "/kaggle/input/datasets/gokulrocky/hyperkvasir-dataset-first-half-and-and-ld-dataset",
    "/kaggle/input/datasets/gokulrocky/hyperkvasir-labeled-videos-part2-001",
    "/kaggle/input/datasets/gokulrocky/om-finalkaggle-upload/weights",
    "/kaggle/input/datasets/gokulrocky/polypdb-polyp-raw/PolypDB/PolypDB/PolypDB_center_wise",
    "/kaggle/input/datasets/gokulrocky/polypdb-polyp-raw/PolypDB/PolypDB/PolypDB_center_wise/BKAI/WLI",
    "/kaggle/input/datasets/gokulrocky/chakratransformer-weights/model_v99.pth",
    "/kaggle/input/datasets/gokulrocky/polypdb-polyp-raw/PolypDB/PolypDB/PolypDB_center_wise/BKAI/BLI/images/PKHL_15 220831_201026_BN032_027.jpg",
]
# You can also load paths dynamically instead of hardcoding, e.g.:
#   CANDIDATE_PATHS = open("/kaggle/working/paths.txt").read().splitlines()
# or pass them in from another notebook cell as a Python list.


def classify_error(exc):
    """Turn an OSError into a short, human-readable reason string."""
    if isinstance(exc, PermissionError):
        return "PERMISSION DENIED"
    if isinstance(exc, FileNotFoundError):
        return "NOT FOUND"
    if isinstance(exc, NotADirectoryError):
        return "NOT A DIRECTORY (path component is a file)"
    if isinstance(exc, OSError):
        # Covers network/mount issues (e.g. stale NFS handle, I/O error)
        if exc.errno == errno.EIO:
            return "I/O ERROR (possible unmounted/stale dataset attach)"
        if exc.errno == errno.ENOTCONN:
            return "NOT CONNECTED (network path unreachable)"
        return f"OS ERROR ({exc.errno}: {exc.strerror})"
    return f"UNKNOWN ERROR ({type(exc).__name__}: {exc})"


def check_path(path):
    """
    Validate a single path. Returns a tuple:
    (status: "OK" | "MISSING", detail: str)

    Deliberately defensive: catches malformed strings, permission
    issues, and I/O errors on flaky mounts rather than letting the
    whole script crash on one bad entry.
    """
    # Guard against garbage input (None, empty string, wrong type, etc.)
    if not isinstance(path, str) or not path.strip():
        return "MISSING", "invalid path (empty or non-string)"

    path = path.strip()

    try:
        exists = os.path.exists(path)
    except (OSError, ValueError) as e:
        # e.g. embedded null bytes, malformed UNC path, unreachable mount
        return "MISSING", classify_error(e)

    if not exists:
        return "MISSING", "does not exist"

    # Path exists -- now confirm we can actually read it (permissions,
    # stale mounts, etc. can make os.path.exists True but access fail).
    try:
        if os.path.isdir(path):
            os.listdir(path)  # forces a real read, not just a stat
            kind = "dir"
        else:
            # Confirm the file is stat-able and has a readable mode
            st = os.stat(path)
            if not stat.S_ISREG(st.st_mode) and not os.path.islink(path):
                kind = "special file"
            else:
                # Try opening a single byte to confirm real read access
                with open(path, "rb") as f:
                    f.read(1)
                kind = "file"
    except PermissionError as e:
        return "MISSING", classify_error(e)
    except OSError as e:
        return "MISSING", classify_error(e)

    return "OK", kind


def run_validation(paths):
    """Run check_path over a list of paths and print a clean report."""
    # De-duplicate while preserving order
    seen = set()
    unique_paths = []
    for p in paths:
        key = p.strip() if isinstance(p, str) else p
        if key not in seen:
            seen.add(key)
            unique_paths.append(p)

    print("=" * 72)
    print(f"VALIDATING {len(unique_paths)} PATH(S)")
    print("=" * 72)

    ok_count = 0
    missing_count = 0
    missing_list = []

    for path in unique_paths:
        status, detail = check_path(path)
        if status == "OK":
            ok_count += 1
            print(f"OK: {path}  ({detail})")
        else:
            missing_count += 1
            missing_list.append(path)
            print(f"MISSING: {path}  -- {detail}")

    print("\n" + "=" * 72)
    print(f"SUMMARY: {ok_count} OK, {missing_count} MISSING (of {len(unique_paths)} total)")
    print("=" * 72)

    if missing_list:
        print("\nMISSING PATHS (copy/paste to re-check after fixing attachments):")
        for p in missing_list:
            print(f"    \"{p}\",")

    return {"ok": ok_count, "missing": missing_count, "missing_paths": missing_list}


if __name__ == "__main__":
    results = run_validation(CANDIDATE_PATHS)

VALIDATING 15 PATH(S)
OK: /kaggle/input/datasets/gokulrocky/chakramodel-evaluation-datasets/kvasir-seg/images  (dir)
OK: /kaggle/input/datasets/gokulrocky/chakramodel-kaggle-code  (dir)
MISSING: /kaggle/input/datasets/gokulrocky/chakratransformer-weights  -- does not exist
MISSING: /kaggle/input/datasets/gokulrocky/cvc-sample-video  -- does not exist
OK: /kaggle/input/datasets/gokulrocky/endoscene-cvc300-polyp-raw-dataset  (dir)
OK: /kaggle/input/datasets/gokulrocky/final-om-evlautation-upload  (dir)
OK: /kaggle/input/datasets/gokulrocky/finalmuruga-harae  (dir)
MISSING: /kaggle/input/datasets/gokulrocky/hperkvasir-labeled-videos-part2-002  -- does not exist
OK: /kaggle/input/datasets/gokulrocky/hyperkvasir-dataset-first-half-and-and-ld-dataset  (dir)
MISSING: /kaggle/input/datasets/gokulrocky/hyperkvasir-labeled-videos-part2-001  -- does not exist
OK: /kaggle/input/datasets/gokulrocky/om-finalkaggle-upload/weights  (dir)
OK: /kaggle/input/datasets/gokulrocky/polypdb-polyp-raw/PolypDB/

## 2. Load Weights

In [ ]:
# Link Weights - Updated to match the real diagnostic path
WEIGHTS_SRC = '/kaggle/input/datasets/gokulraj324/chakramodel-weightsupdated4'
!mkdir -p weights
if os.path.exists(WEIGHTS_SRC):
    !cp -r {WEIGHTS_SRC}/* weights/
    print("Weights loaded successfully.")
else:
    print(f"WARNING: Weights path {WEIGHTS_SRC} not found.")

## 3. Video Inference & Temporal Proof (FPS Logging)

In [ ]:
import glob

# Find any mp4 videos attached to the notebook
video_files = glob.glob("/kaggle/input/**/*.mp4", recursive=True)
if not video_files:
    print("WARNING: No mp4 videos found in /kaggle/input.")
else:
    print(f"Found {len(video_files)} video files. Running inference on the first 5 videos...\n")
    for vid in video_files[:5]:
        print(f"Evaluating: {vid}")
        !python src/infer_stream.py --source "{vid}" --model combo1 --weights weights/combo1_best.pth --yolo weights/best.pt

## 4. Full-Cohort Cross-Dataset Evaluation (OOD Truth)

In [ ]:
import sys
import cv2
import numpy as np
import json
from tqdm.auto import tqdm

sys.path.append('/kaggle/working/chakramodel/src')
try:
    from chakranet_segmenter import ChakraNet
    from metrics.seg_metrics import dice as binary_dice_coefficient, iou as binary_iou
    segmenter = ChakraNet(device="cpu")
except Exception as e:
    print(f"Failed to load segmenter: {e}")
    segmenter = None

def eval_folder(img_dir, mask_dir):
    img_paths = [os.path.join(img_dir, f) for f in os.listdir(img_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    results = []
    for img_path in tqdm(img_paths, desc=os.path.basename(os.path.dirname(img_dir))):
        basename = os.path.basename(img_path)
        mask_path = os.path.join(mask_dir, basename)
        if not os.path.exists(mask_path):
            mask_path = os.path.join(mask_dir, os.path.splitext(basename)[0] + '.png')
            if not os.path.exists(mask_path):
                mask_path = os.path.join(mask_dir, os.path.splitext(basename)[0] + '.jpg')
        if not os.path.exists(mask_path): continue
        img = cv2.imread(img_path)
        gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if img is None or gt_mask is None: continue
        gt_mask = (gt_mask > 127).astype(np.uint8)
        pred_mask, _, _, _ = segmenter.segment_roi(img, threshold=0.45, mc_passes=1)
        if pred_mask is None: continue
        pred_mask = cv2.resize(pred_mask, (gt_mask.shape[1], gt_mask.shape[0]), interpolation=cv2.INTER_NEAREST)
        pred_mask_bin = (pred_mask > 127).astype(np.uint8)
        results.append({"image": basename, "dice": binary_dice_coefficient(pred_mask_bin, gt_mask), "iou": binary_iou(pred_mask_bin, gt_mask)})
    if not results: return {"images": 0, "mean_dice": 0, "mean_iou": 0}
    return {"images": len(results), "mean_dice": float(np.mean([r["dice"] for r in results])), "mean_iou": float(np.mean([r["iou"] for r in results]))}

final_metrics = {}
if segmenter:
    print("\nScanning /kaggle/input for image/mask pairs...")
    for root, dirs, files in os.walk("/kaggle/input"):
        if "images" in dirs and "masks" in dirs:
            dataset_name = os.path.basename(root)
            # Avoid duplicate checks if nested
            if dataset_name not in final_metrics:
                print(f"\nEvaluating found dataset: {dataset_name}...")
                final_metrics[dataset_name] = eval_folder(os.path.join(root, "images"), os.path.join(root, "masks"))

    !mkdir -p results
    with open("results/final_proof_eval.json", "w") as f:
        json.dump(final_metrics, f, indent=4)
    print("\nFINAL PROOF METRICS:\n", json.dumps(final_metrics, indent=4))


## 5. Artifact Packaging

In [ ]:
%cd /kaggle/working
!zip -r final_proof_artifacts.zip chakramodel/results chakramodel/outputs > /dev/null
print("\nDone! You can now download final_proof_artifacts.zip from the output data panel.")